# Building a Minimal Secure MCP Server

## Scenario and safety boundary
This notebook uses the official Python SDK to exercise a real MCP server in memory. The single-tenant fixture is offline and credential-free. It can read bounded ticket data and compute a reply proposal, but it has no send, write, filesystem, shell, code, or arbitrary-network capability.

In [ ]:
import json
import runpy
from pathlib import Path

from mcp import Client

module = runpy.run_path(Path('lab.py'))
mcp = module['mcp']

## 1. Run the complete protocol scenario
`Client(mcp)` uses the real SDK protocol layer without opening a subprocess or port. It negotiates, discovers, calls tools, reads the resource, and renders the prompt.

In [ ]:
evidence, payload = await module['run_scenario']()
assert evidence.protocol_version == '2026-07-28'
assert evidence.tools == ('ticket.list_open', 'ticket.propose_reply', 'ticket.read')
assert evidence.external_effect_count == 0
evidence

## 2. Inspect the generated contract
The SDK derives bounded inputs from constrained annotations and closed outputs from Pydantic models. Tool annotations are honest UI hints, not authorization.

In [ ]:
async with Client(mcp) as client:
    tools = {tool.name: tool for tool in (await client.list_tools()).tools}

read_schema = tools['ticket.read'].input_schema
proposal_output = tools['ticket.propose_reply'].output_schema
assert read_schema['properties']['ticket_id']['maxLength'] == 63
assert proposal_output['additionalProperties'] is False
assert all(tool.annotations.read_only_hint for tool in tools.values())
read_schema, proposal_output

## 3. Attack the argument boundary
The exact-field middleware rejects extra or missing fields. The SDK rejects bad types and length constraints. None of these malformed calls reaches an application handler.

In [ ]:
module['reset_runtime_evidence']()
async with Client(mcp) as client:
    extra = await client.call_tool('ticket.read', {'ticket_id': 'acme-7', 'debug': True})
    wrong_type = await client.call_tool('ticket.read', {'ticket_id': 7})
    oversized = await client.call_tool(
        'ticket.propose_reply', {'ticket_id': 'acme-7', 'body': 'x' * 501}
    )
assert extra.is_error and wrong_type.is_error and oversized.is_error
assert sum(module['HANDLER_CALLS'].values()) == 0
[extra.content[0].text, wrong_type.content[0].text, oversized.content[0].text]

## 4. Deny forbidden resources without confirming existence
The fixture contains `globex-9`; `acme-404` does not exist. Both receive the same tool error and no structured ticket data.

In [ ]:
module['reset_runtime_evidence']()
async with Client(mcp) as client:
    cross_tenant = await client.call_tool('ticket.read', {'ticket_id': 'globex-9'})
    unknown = await client.call_tool('ticket.read', {'ticket_id': 'acme-404'})
assert cross_tenant.is_error and unknown.is_error
assert cross_tenant.structured_content is None and unknown.structured_content is None
cross_tenant.content[0].text, unknown.content[0].text

## 5. Proposal is not execution
The proposal is structured and digest-bound to its exact content, but it grants no approval and triggers no side effect. A same-length body change produces a different digest.

In [ ]:
module['reset_runtime_evidence']()
async with Client(mcp) as client:
    first = await client.call_tool(
        'ticket.propose_reply', {'ticket_id': 'acme-7', 'body': 'Allow access'}
    )
    changed = await client.call_tool(
        'ticket.propose_reply', {'ticket_id': 'acme-7', 'body': 'Deny access!'}
    )
assert first.structured_content['action_digest'] != changed.structured_content['action_digest']
assert first.structured_content['executed'] is False
assert module['EXTERNAL_EFFECTS'] == []
first.structured_content

## 6. Resources and prompts carry provenance, not authority
The resource has an exact URI, policy version, digest, and trust label. The prompt has a visible template version and remains untrusted text. Neither calls a tool.

In [ ]:
module['reset_runtime_evidence']()
async with Client(mcp) as client:
    resource_result = await client.read_resource('support://acme/policy/2026-09-21')
    prompt_result = await client.get_prompt('support_summary', {'ticket_id': 'acme-7'})
policy = json.loads(resource_result.contents[0].text)
prompt_text = prompt_result.messages[0].content.text
assert policy['content_trust'] == 'untrusted-server-resource'
assert 'trust=untrusted-template' in prompt_text
assert sum(module['HANDLER_CALLS'].values()) == 0
policy, prompt_text

## 7. Inspect redacted evidence
Application audit records contain request, tool, resource, decision, reason, digest, and policy version. They omit ticket summaries and reply bodies.

In [ ]:
module['reset_runtime_evidence']()
secret_body = 'temporary-secret-value'
async with Client(mcp) as client:
    proposal = await client.call_tool(
        'ticket.propose_reply', {'ticket_id': 'acme-7', 'body': secret_body}
    )
audit_json = json.dumps([event.__dict__ for event in module['AUDIT_EVENTS']])
assert secret_body not in audit_json
assert module['AUDIT_EVENTS'][-1].argument_digest == proposal.structured_content['action_digest']
module['AUDIT_EVENTS'][-1]

## Evaluation and production upgrade
Report malformed-call pre-handler rate, forbidden disclosure rate, output conformance, authorized-call success, effect escape count, and audit coverage with explicit denominators. The in-memory path proves protocol behavior, not process isolation, TLS, authentication, proxy limits, downstream idempotency, or model quality. Add subprocess and Streamable HTTP staging tests before deployment.

## Reflection
Which properties came from the SDK, middleware, Pydantic model, deployment policy, or application handler? Why can a valid schema, `readOnlyHint`, or proposal digest never authorize a reply to be sent?